# EXP-013 — SVM cân bằng trọng số lớp trên WESAD

**Người thực hiện:** Nam  
**Nhóm:** 03 — Class Balancing  
**Trạng thái:** Chưa chạy trong bản mã nguồn phát hành.

## 1. Mục tiêu và giả thuyết

Kiểm tra việc dùng `class_weight="balanced"` trong SVM RBF có cải thiện Macro-F1 và Balanced Accuracy so với EXP-004 (SVM mặc định) trên cùng WESAD physiological68 hay không.

## 2. Thiết kế thí nghiệm

- Bộ G2: 68 đặc trưng, 15 người, 6.189 cửa sổ theo bộ được nhóm nghiệm thu.
- Outer LOSO: huấn luyện 14 người, kiểm tra người còn lại (15 folds × 2 tasks = 30).
- `SimpleImputer` và `StandardScaler` được fit trên train mỗi fold.
- SVM RBF mặc định `C=1.0`, `gamma="scale"`; chỉ đổi `class_weight="balanced"`.
- Không tuning, không oversampling. Nhãn nhị phân 0/1 và ba lớp 1/2/3.

## 3. Khởi tạo môi trường và đọc cấu hình

In [ ]:
from pathlib import Path
import sys, json, hashlib, subprocess
import pandas as pd
import yaml
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p/'scripts/run_experiment.py').is_file() and (p/'src/wesad/models.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Không tìm thấy wesad-stress-detection. Hãy mở notebook trong repo đúng.')
print('ROOT:', ROOT)
print('Python:', sys.executable)

def sha256(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(1024*1024), b''):
            digest.update(block)
    return digest.hexdigest()

def execute(args):
    command=[sys.executable,*args]
    print('Lệnh:', ' '.join(map(str,command)))
    proc=subprocess.run(command,cwd=ROOT,capture_output=True,text=True,encoding='utf-8',errors='replace')
    print('Return code:',proc.returncode)
    if proc.stdout: print('STDOUT: \n'+proc.stdout[-12000:])
    if proc.stderr: print('STDERR: \n'+proc.stderr[-12000:])
    if proc.returncode: raise RuntimeError(f'Chạy thất bại ({proc.returncode}). Xem STDERR ở trên.')
    return proc

CONFIG_PATH=ROOT/'experiments/03_balancing/EXP-013_SVM/config.yaml'
EXPORT_DIR=CONFIG_PATH.parent
MAIN_DIR=ROOT/'experiments/03_balancing/EXP-013_SVM/run_outputs/main'
assert CONFIG_PATH.is_file()
print(CONFIG_PATH)

## 4. Kiểm tra dữ liệu và checksum

In [ ]:
CONFIG=yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
DATA=ROOT/CONFIG['features']; SCHEMA=ROOT/CONFIG['schema']
for f in [DATA,SCHEMA]:
    if not f.is_file(): raise FileNotFoundError(f'Thiếu dữ liệu G2: {f}')
actual=sha256(DATA)
assert actual==CONFIG['feature_sha256'],f'Không khớp checksum: {actual}'
sys.path.insert(0,str(ROOT/'src'))
from wesad.data import load_feature_table
frame, features=load_feature_table(DATA,SCHEMA)
assert len(features)==68 and frame.subject_id.nunique()==15
print('Số cửa sổ:',len(frame),'| Số người:',frame.subject_id.nunique(),'| Số đặc trưng:',len(features))
display(frame.label_original.value_counts().sort_index().rename('Số cửa sổ').to_frame())


## 5. Kiểm tra cấu hình SVM

In [ ]:
from wesad.models import build_pipeline
assert CONFIG['models']==['svm'] and CONFIG['balance'] is True and CONFIG['nested'] is False
balanced=build_pipeline('svm',CONFIG)
normal_config=dict(CONFIG);normal_config['balance']=False
baseline=build_pipeline('svm',normal_config)
print('EXP-004 baseline:',baseline['model'])
print('EXP-013 class-balanced:',balanced['model'])
assert balanced['model'].get_params()['class_weight']=='balanced'


## 6. R01 — Chạy thử một fold S2

Bật `RUN_QUICK=True` nếu muốn chạy thử. Output quick nằm ngoài thư mục báo cáo chính, không được xem là 15-fold chính thức.

In [ ]:
RUN_QUICK=False
if RUN_QUICK:
    # Quick dùng --outer-subjects và tách output để không lẫn main.
    quick_config=dict(CONFIG)
    quick_config['output_dir']='outputs/notebook_research/EXP-013/quick_s2'
    quick_path=EXPORT_DIR/'quick_s2_config.yaml'
    quick_path.write_text(yaml.safe_dump(quick_config,sort_keys=False,allow_unicode=True),encoding='utf-8')
    execute(['scripts/run_experiment.py','--config',str(quick_path),'--outer-subjects','S2'])
else: print('R01 chưa chạy: bật RUN_QUICK=True nếu cần.')


## 7. R02 — Chạy chính thức và xuất CSV/PNG

Chạy 30 LOSO fold. Sau khi runner hoàn tất, tự tạo các file `results.csv`, `summary.csv`, `predictions.csv`, `result_provenance.json`, ảnh PNG **ngay trong thư mục EXP-013_SVM**, giống EXP-007.

In [ ]:
RUN_MAIN=False
if RUN_MAIN:
    execute(['scripts/run_experiment.py','--config',str(CONFIG_PATH),'--resume'])
    execute(['scripts/report_exp013.py'])
else: print('Chưa chạy chính thức. Đặt RUN_MAIN=True.')


## 8. Đọc và kiểm chứng kết quả

Nếu đã chạy main trước đó nhưng thiếu CSV/PNG, bật `RUN_REPORT=True` để xuất báo cáo mà không huấn luyện lại.

In [ ]:
RUN_REPORT=False
if RUN_REPORT:
    execute(['scripts/report_exp013.py'])
summary_path=EXPORT_DIR/'summary.csv'
if summary_path.exists():
    display(pd.read_csv(summary_path))
    display(pd.read_csv(EXPORT_DIR/'results.csv').head(10))
else: print('Chưa có summary.csv. Chạy main và export trước.')


## 9. Phân tích biểu đồ và so sánh EXP-004

In [ ]:
from IPython.display import Image
figures=sorted((EXPORT_DIR/'figures').glob('*.png'))
print('Số ảnh:',len(figures))
for f in figures: print('-',f.name)
comparison=EXPORT_DIR/'comparison_exp004.csv'
if comparison.is_file():
    display(pd.read_csv(comparison).groupby('task')[['delta_macro_f1','delta_balanced_accuracy']].agg(['mean','std']))
else: print('Chưa tạo comparison_exp004.csv vì thiếu bằng chứng baseline tương thích.')


## 10. Hạn chế và kiểm tra nguồn

Số người chỉ 15; các cửa sổ chồng lấn 55 giây. `class_weight=balanced` chỉ điều chỉnh trọng số lỗi, không sinh thêm cửa sổ. Không kết luận có cải thiện nếu chưa có chênh lệch ghép cặp với EXP-004 trên cùng 15 người, cùng dữ liệu và nhãn.

## 11. Nhận xét và kết luận

**Điền sau khi có kết quả:**

- Nhị phân: Macro-F1 mean ± std = …; Balanced Accuracy = ….
- Ba lớp: Macro-F1 mean ± std = …; Balanced Accuracy = ….
- So với EXP-004: chênh lệch trung bình và người tăng/giảm nhiều nhất từ `comparison_exp004.csv` (nếu có).
- Kết luận cân bằng trọng số có cải thiện hay không phải dựa trên dữ liệu thật. Không tự điền số liệu.